# Post Pandemic Regime Shifts in Labor Market: Pulling Data from FRED

## Purpose

## Research Context

## Inputs and Outputs

## Imports and Configuration

In [ ]:
import os
import re
from pathlib import Path
import requests

import numpy as np
import pandas as pd

from fredapi import Fred

In [28]:
repo_root = Path.home() / "Documents" / "Coding" / "ML_LaborTightnessRegimeShift"
# repo_root = Path(r"C:\Users\sumai\Documents\ML_LaborTightnessRegimeShift")

data_root = repo_root / "data"
raw_dir = data_root / "raw"
external_dir = raw_dir / "external_sources"
fred_dir = raw_dir / "fred"

fred_dir.mkdir(parents=True, exist_ok=True)
external_dir.mkdir(parents=True, exist_ok=True)

RAW_API_KEY = """

"""

In [29]:
api_key = re.sub(r"\s+", "", RAW_API_KEY)

assert api_key != "", "API key is empty."
assert " " not in api_key, "API key still contains spaces."
assert "\n" not in api_key and "\r" not in api_key and "\t" not in api_key, "API key still contains whitespace."

fred = Fred(api_key=api_key)

start_date = "2000-01-01"
end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

In [30]:
series_map = {
    "job_openings_level": "JTSJOL",
    "job_openings_level_sa": "JTSJOLSL",
    "hires_rate": "JTSHIR",
    "hires_level": "JTSHIL",
    "quits_rate": "JTSQUR",
    "quits_level": "JTSQUL",
    "total_separations_rate": "JTSTSR",
    "total_separations_level": "JTSTSL",
    "layoffs_discharges_level": "JTSLDL",
    "layoffs_discharges_rate": "JTSLDR",
    "unemployment_rate": "UNRATE",
    "unemployment_level": "UNEMPLOY",
    "payrolls_total_nonfarm": "PAYEMS",
    "payrolls_total_private": "USPRIV",
    "payrolls_manufacturing": "MANEMP",
    "payrolls_service_providing": "SRVPRD",
    "payrolls_construction": "USCONS",
    "prime_age_lfpr": "LNS11300060",
    "labor_force_participation_rate": "CIVPART",
    "employment_population_ratio": "EMRATIO",
    "u6_unemployment_rate": "U6RATE",
    "prime_age_unemployment_rate": "LNS13025703",
    "average_weeks_unemployed": "LNS12300060",
    "avg_hourly_earnings_total_private": "CES0500000003",
    "avg_weekly_earnings_total_private": "CES0500000008",
    "avg_weekly_hours_total_private": "CES0500000007",
    "avg_weekly_earnings_manufacturing": "CES3000000008",
    "avg_hourly_earnings_manufacturing": "CES3000000003",
    "cpi_all_items": "CPIAUCSL",
    "cpi_core": "CPILFESL",
    "cpi_all_items_less_shelter": "CUSR0000SA0L2",
    "cpi_services_less_rent_of_shelter": "CUUR0000SASL2RS",
    "cpi_services_less_energy_services": "CUSR0000SASLE",
    "cpi_shelter": "CUSR0000SEHA",
    "pce_price_index": "PCEPI",
    "pce_core_price_index": "PCEPILFE",
    "personal_saving_rate": "PMSAVE",
    "real_disposable_personal_income": "DSPIC96",
    "real_personal_income": "RPI",
    "real_personal_income_less_transfers": "W875RX1",
    "retail_sales_advance": "RSAFS",
    "real_retail_sales": "RRSFS",
    "industrial_production": "INDPRO",
    "industrial_production_manufacturing": "IPMAN",
    "capacity_utilization": "CUMFNS",
    "housing_starts": "HOUST",
    "building_permits": "PERMIT",
    "new_one_family_houses_sold": "HSN1F",
    "consumer_sentiment": "UMCSENT",
    "ism_pmi": "NAPM",
    "durable_goods_orders": "DGORDER",
    "capital_goods_orders_ex_aircraft": "NEWORDER",
    "commercial_industrial_loans": "BUSLOANS",
    "total_bank_credit": "TOTBKCR",
    "m2_money_stock": "M2SL",
    "treasury_1m": "DGS1MO",
    "treasury_3m": "DGS3MO",
    "treasury_2y": "DGS2",
    "treasury_10y": "GS10",
    "fed_funds_rate": "FEDFUNDS",
    "aaa_corporate_yield": "AAA",
    "baa_corporate_yield": "BAA",
    "high_yield_oas": "BAMLH0A0HYM2",
    "bbb_oas": "BAMLC0A4CBBB",
    "breakeven_5y": "T5YIE",
    "forward_5y5y_inflation": "T5YIFR",
    "michigan_inflation_1y": "MICH",
    "michigan_inflation_5y": "MICH5Y",
    "recession_indicator": "USREC",
}

In [31]:
def get_series_with_info(series_id, col_name, start_date=start_date, end_date=end_date):
    series = fred.get_series(
        series_id,
        observation_start=start_date,
        observation_end=end_date,
    )

    series = pd.Series(series, name=col_name)
    series.index = pd.to_datetime(series.index)

    try:
        info = fred.get_series_info(series_id)
    except Exception:
        info = pd.Series(dtype="object")

    return series, info

def convert_to_month_start(series, freq_code):
    freq_code = str(freq_code).strip().upper()

    if freq_code == "M":
        monthly = series.copy()
    elif freq_code in {"D", "W", "BW"}:
        monthly = series.resample("MS").mean()
    elif freq_code == "Q":
        monthly = series.resample("MS").ffill()
    else:
        monthly = series.resample("MS").mean()

    monthly.index = pd.to_datetime(monthly.index)
    monthly = monthly.sort_index()
    return monthly

In [32]:
series_list = []
meta_rows = []
failed_rows = []

for col_name, series_id in series_map.items():
    try:
        series, info = get_series_with_info(series_id, col_name)

        if series.empty:
            failed_rows.append({
                "column_name": col_name,
                "series_id": series_id,
                "error": "empty_series",
            })
            continue

        freq_code = info.get("frequency_short", "")
        monthly = convert_to_month_start(series, freq_code)
        monthly.name = col_name

        series_list.append(monthly)

        meta_rows.append({
            "column_name": col_name,
            "series_id": series_id,
            "title": info.get("title", ""),
            "frequency_short": str(freq_code).strip().upper(),
            "units": info.get("units", ""),
            "seasonal_adjustment": info.get("seasonal_adjustment", ""),
            "last_updated": info.get("last_updated", ""),
            "observation_start": info.get("observation_start", ""),
            "observation_end": info.get("observation_end", ""),
            "notes": info.get("notes", ""),
        })

    except Exception as e:
        failed_rows.append({
            "column_name": col_name,
            "series_id": series_id,
            "error": str(e),
        })

if not series_list:
    raise RuntimeError("No downloads, maybe API error?")

In [43]:
fred_raw = pd.concat(series_list, axis=1, sort=False).sort_index()
fred_raw = fred_raw.loc[
    (fred_raw.index >= pd.Timestamp(start_date)) &
    (fred_raw.index <= pd.Timestamp(end_date))
].copy()

fred_raw.index.name = "date"

fred_meta = pd.DataFrame(meta_rows).sort_values("column_name").reset_index(drop=True)
fred_failed = pd.DataFrame(failed_rows)

fred_missing = (
    fred_raw.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)

fred_missing["missing_pct"] = fred_missing["missing_count"] / len(fred_raw)

In [44]:
fred_csv_path = fred_dir / "fred_raw_monthly.csv"
fred_parquet_path = fred_dir / "fred_raw_monthly.parquet"
fred_meta_path = fred_dir / "fred_metadata.csv"
fred_failed_path = fred_dir / "fred_failed_series.csv"
fred_missing_path = fred_dir / "fred_missing_summary.csv"

fred_raw.to_csv(fred_csv_path)
fred_meta.to_csv(fred_meta_path, index=False)
fred_failed.to_csv(fred_failed_path, index=False)
fred_missing.to_csv(fred_missing_path)

In [45]:
print("FRED Raw Shape:", fred_raw.shape)
print("FRED Date Range:", fred_raw.index.min(), "to", fred_raw.index.max())
print("CSV:", fred_csv_path)
print("Metadata:", fred_meta_path)
print("Failed Series:", fred_failed_path)
print("Missing Summary:", fred_missing_path)

display(fred_raw.head())
display(fred_meta.head(20))
display(fred_failed.head(20))
display(fred_missing.head(25))

FRED Raw Shape: (315, 58)
FRED Date Range: 2000-01-01 00:00:00 to 2026-03-01 00:00:00
CSV: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/fred/fred_raw_monthly.csv
Metadata: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/fred/fred_metadata.csv
Failed Series: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/fred/fred_failed_series.csv
Missing Summary: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/fred/fred_missing_summary.csv


,job_openings_level,hires_rate,hires_level,quits_rate,quits_level,total_separations_rate,total_separations_level,layoffs_discharges_level,layoffs_discharges_rate,unemployment_rate,...,commercial_industrial_loans,total_bank_credit,m2_money_stock,treasury_1m,treasury_3m,treasury_2y,treasury_10y,fed_funds_rate,aaa_corporate_yield,baa_corporate_yield
date,,,,,,,,,,,,,,,,,,,,,
2000-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,...,1003.1482,4535.67470,4667.6,NaN,5.499000,6.440000,6.66,5.45,7.78,8.33
2000-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.1,...,1016.2395,4554.79750,4680.9,NaN,5.727000,6.610500,6.52,5.73,7.68,8.29
2000-03-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,...,1026.2416,4609.44284,4711.7,NaN,5.863913,6.528261,6.26,5.85,7.68,8.37
2000-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.8,...,1035.2058,4653.21385,4767.8,NaN,5.821579,6.403684,5.99,6.02,7.64,8.40
2000-05-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,...,1051.7891,4703.24964,4755.7,NaN,5.994545,6.809545,6.44,6.27,7.99,8.90


,column_name,series_id,title,frequency_short,units,seasonal_adjustment,last_updated,observation_start,observation_end,notes
0,aaa_corporate_yield,AAA,Moody's Seasoned Aaa Corporate Bond Yield,M,Percent,Not Seasonally Adjusted,2026-03-02 10:17:47-06,1919-01-01,2026-02-01,These instruments are based on bonds with matu...
1,average_weeks_unemployed,LNS12300060,Employment-Population Ratio - 25-54 Yrs.,M,Percent,Seasonally Adjusted,2026-03-06 08:15:07-06,1948-01-01,2026-02-01,
2,avg_hourly_earnings_manufacturing,CES3000000003,"Average Hourly Earnings of All Employees, Manu...",M,Dollars per Hour,Seasonally Adjusted,2026-03-06 08:16:26-06,2006-03-01,2026-02-01,The series comes from the 'Current Employment ...
3,avg_hourly_earnings_total_private,CES0500000003,"Average Hourly Earnings of All Employees, Tota...",M,Dollars per Hour,Seasonally Adjusted,2026-03-06 08:16:10-06,2006-03-01,2026-02-01,The series comes from the 'Current Employment ...
4,avg_weekly_earnings_manufacturing,CES3000000008,Average Hourly Earnings of Production and Nons...,M,Dollars per Hour,Seasonally Adjusted,2026-03-06 08:16:29-06,1939-01-01,2026-02-01,Production and related employees include worki...
5,baa_corporate_yield,BAA,Moody's Seasoned Baa Corporate Bond Yield,M,Percent,Not Seasonally Adjusted,2026-03-02 10:17:45-06,1919-01-01,2026-02-01,These instruments are based on bonds with matu...
6,building_permits,PERMIT,New Privately-Owned Housing Units Authorized i...,M,Thousands of Units,Seasonally Adjusted Annual Rate,2026-03-12 07:32:54-05,1960-01-01,2026-01-01,"Starting with the 2005-02-16 release, the seri..."
7,capacity_utilization,CUMFNS,Capacity Utilization: Manufacturing (SIC),M,Percent,Seasonally Adjusted,2026-03-16 08:19:53-05,1948-01-01,2026-02-01,explanatory notes (https://www.federalreserve....
8,capital_goods_orders_ex_aircraft,NEWORDER,Manufacturers' New Orders: Nondefense Capital ...,M,Millions of Dollars,Seasonally Adjusted,2026-03-13 07:34:18-05,1992-02-01,2026-01-01,"Effective May 21, 2001, data were reconstructe..."
9,commercial_industrial_loans,BUSLOANS,"Commercial and Industrial Loans, All Commercia...",M,Billions of U.S. Dollars,Seasonally Adjusted,2026-03-13 15:47:21-05,1947-01-01,2026-02-01,data source (https://www.federalreserve.gov/ap...


,column_name,series_id,error
0,job_openings_level_sa,JTSJOLSL,Bad Request. The series does not exist.
1,avg_weekly_earnings_total_private,CES0500000008,Bad Request. The series does not exist.
2,avg_weekly_hours_total_private,CES0500000007,Bad Request. The series does not exist.
3,ism_pmi,NAPM,Bad Request. The series does not exist.
4,high_yield_oas,BAMLH0A0HYM2,Too Many Requests. Exceeded Rate Limit
5,bbb_oas,BAMLC0A4CBBB,Too Many Requests. Exceeded Rate Limit
6,breakeven_5y,T5YIE,Too Many Requests. Exceeded Rate Limit
7,forward_5y5y_inflation,T5YIFR,Too Many Requests. Exceeded Rate Limit
8,michigan_inflation_1y,MICH,Too Many Requests. Exceeded Rate Limit
9,michigan_inflation_5y,MICH5Y,Too Many Requests. Exceeded Rate Limit


,missing_count,missing_pct
avg_hourly_earnings_manufacturing,75,0.238095
avg_hourly_earnings_total_private,75,0.238095
treasury_1m,18,0.057143
job_openings_level,13,0.041270
hires_rate,13,0.041270
hires_level,13,0.041270
quits_rate,13,0.041270
quits_level,13,0.041270
total_separations_rate,13,0.041270
total_separations_level,13,0.041270


In [46]:
external_urls = {
    "atlanta_fed_wage_growth_tracker": "https://www.atlantafed.org/-/media/Project/Atlanta/FRBA/Documents/datafiles/chcs/wage-growth-tracker/wage-growth-data.xlsx",
    "philly_fed_ruc_vintages": "https://www.philadelphiafed.org/-/media/FRBP/Assets/Surveys-And-Data/real-time-data/data-files/xlsx/rucQvMd.xlsx",
    "philly_fed_cpi_vintages": "https://www.philadelphiafed.org/-/media/FRBP/Assets/Surveys-And-Data/real-time-data/data-files/xlsx/cpiQvMd.xlsx",
    "philly_fed_spf_inflation": "https://www.philadelphiafed.org/-/media/FRBP/Assets/Surveys-And-Data/survey-of-professional-forecasters/historical-data/Inflation.xlsx",
}

In [47]:
downloaded_files = {}

for name, url in external_urls.items():
    out_path = external_dir / f"{name}.xlsx"

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    with open(out_path, "wb") as f:
        f.write(response.content)

    downloaded_files[name] = out_path

print("Downloaded External Files:")
for name, path in downloaded_files.items():
    print(name, "->", path)

Downloaded External Files:
atlanta_fed_wage_growth_tracker -> /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/external_sources/atlanta_fed_wage_growth_tracker.xlsx
philly_fed_ruc_vintages -> /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/external_sources/philly_fed_ruc_vintages.xlsx
philly_fed_cpi_vintages -> /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/external_sources/philly_fed_cpi_vintages.xlsx
philly_fed_spf_inflation -> /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/external_sources/philly_fed_spf_inflation.xlsx


In [48]:
key_raw_dir = external_dir / "key_raw_tables"
key_raw_dir.mkdir(parents=True, exist_ok=True)

In [49]:
atl_path = downloaded_files["atlanta_fed_wage_growth_tracker"]
ruc_path = downloaded_files["philly_fed_ruc_vintages"]
cpi_path = downloaded_files["philly_fed_cpi_vintages"]
spf_path = downloaded_files["philly_fed_spf_inflation"]

atl_overall = pd.read_excel(atl_path, sheet_name="data_overall", engine="openpyxl")
atl_overall_12ma = pd.read_excel(atl_path, sheet_name="Overall 12ma", engine="openpyxl")

phl_ruc = pd.read_excel(ruc_path, sheet_name="ruc", engine="openpyxl")
phl_cpi = pd.read_excel(cpi_path, sheet_name="cpi", engine="openpyxl")
spf_inflation = pd.read_excel(spf_path, sheet_name="INFLATION", engine="openpyxl")

print("Raw Shapes")
print(". . .")
print("atl_overall:", atl_overall.shape)
print("atl_overall_12ma:", atl_overall_12ma.shape)
print("phl_ruc:", phl_ruc.shape)
print("phl_cpi:", phl_cpi.shape)
print("spf_inflation:", spf_inflation.shape)

display(atl_overall.head())
display(atl_overall_12ma.head())
display(phl_ruc.head())
display(phl_cpi.head())
display(spf_inflation.head())

Raw Shapes
. . .
atl_overall: (351, 17)
atl_overall_12ma: (352, 4)
phl_ruc: (949, 243)
phl_cpi: (949, 243)
spf_inflation: (225, 5)


/Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/.venv/lib/python3.14/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,"Sources: Current Population Survey, Bureau of Labor Statistics, and Federal Reserve Bank of Atlanta Calculations. \nData updates can be found at https://www.frbatlanta/chcs/wage-growth-tracker. \nSee the 'definitions' tab in this spreadsheet for explanations of series. Wage computed on an hourly basis unless otherwise noted. Reported as 3-month moving averages of monthly series",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,2026-01-12 00:00:00,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,NaT,Overall,Services,Full-time,College degree,Age 25-54,Female,Male,Job Stayer,Job Switcher,Paid Hourly,Overall: Weighted,Overall: Weighted 97,Overall: Weekly Basis,Overall: 25/20 trimmed mean,Lower 1/2 of wage distn,Upper 1/2 of wage distn
1,1997-01-01,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.
2,1997-02-01,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.
3,1997-03-01,4.5,4.6,4.5,4.7,4.4,4.6,4.4,4.1,5.2,4.2,4.9,4.9,4.8,4.5,4.8,4.2
4,1997-04-01,4.6,4.7,4.6,4.6,4.5,4.5,4.6,4.1,5.4,4.3,5,5,4.9,4.5,4.9,4.2


,"Sources: Current Population Survey, Bureau of Labor Statistics, and Federal Reserve Bank of Atlanta Calculations. \nData updates can be found at https://www.frbatlanta/chcs/wage-growth-tracker.",Unnamed: 1,Unnamed: 2,Unnamed: 3
0,The data are 12 month moving averages of month...,NaN,NaN,NaN
1,NaN,Overall,Overall: Weighted,Overall: Weighted 97
2,1997-01-01 00:00:00,.,.,.
3,1997-02-01 00:00:00,.,.,.
4,1997-03-01 00:00:00,.,.,.


,DATE,RUC65Q4,RUC66Q1,RUC66Q2,RUC66Q3,RUC66Q4,RUC67Q1,RUC67Q2,RUC67Q3,RUC67Q4,...,RUC23Q4,RUC24Q1,RUC24Q2,RUC24Q3,RUC24Q4,RUC25Q1,RUC25Q2,RUC25Q3,RUC25Q4,RUC26Q1
0,1947:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1947:02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1947:03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1947:04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1947:05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,DATE,CPI65Q4,CPI66Q1,CPI66Q2,CPI66Q3,CPI66Q4,CPI67Q1,CPI67Q2,CPI67Q3,CPI67Q4,...,CPI23Q4,CPI24Q1,CPI24Q2,CPI24Q3,CPI24Q4,CPI25Q1,CPI25Q2,CPI25Q3,CPI25Q4,CPI26Q1
0,1947:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48
1,1947:02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62
2,1947:03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00
3,1947:04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00
4,1947:05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95


,YEAR,QUARTER,INFPGDP1YR,INFCPI1YR,INFCPI10YR
0,1970,1,NaN,NaN,NaN
1,1970,2,2.9851,NaN,NaN
2,1970,3,3.7037,NaN,NaN
3,1970,4,3.5414,NaN,NaN
4,1971,1,3.5303,NaN,NaN


In [50]:
atl_overall_path = key_raw_dir / "atlanta_fed_wgt_overall_raw.csv"
atl_overall_12ma_path = key_raw_dir / "atlanta_fed_wgt_overall_12ma_raw.csv"
phl_ruc_path = key_raw_dir / "philly_fed_ruc_vintages_raw.csv"
phl_cpi_path = key_raw_dir / "philly_fed_cpi_vintages_raw.csv"
spf_inflation_path = key_raw_dir / "philly_fed_spf_inflation_raw.csv"

atl_overall.to_csv(atl_overall_path, index=False)
atl_overall_12ma.to_csv(atl_overall_12ma_path, index=False)
phl_ruc.to_csv(phl_ruc_path, index=False)
phl_cpi.to_csv(phl_cpi_path, index=False)
spf_inflation.to_csv(spf_inflation_path, index=False)

In [51]:
external_manifest = pd.DataFrame(
    [
        {
            "table_name": "atl_overall",
            "path": str(atl_overall_path),
            "rows": len(atl_overall),
            "cols": atl_overall.shape[1],
        },
        {
            "table_name": "atl_overall_12ma",
            "path": str(atl_overall_12ma_path),
            "rows": len(atl_overall_12ma),
            "cols": atl_overall_12ma.shape[1],
        },
        {
            "table_name": "phl_ruc",
            "path": str(phl_ruc_path),
            "rows": len(phl_ruc),
            "cols": phl_ruc.shape[1],
        },
        {
            "table_name": "phl_cpi",
            "path": str(phl_cpi_path),
            "rows": len(phl_cpi),
            "cols": phl_cpi.shape[1],
        },
        {
            "table_name": "spf_inflation",
            "path": str(spf_inflation_path),
            "rows": len(spf_inflation),
            "cols": spf_inflation.shape[1],
        },
    ]
)



In [52]:
external_manifest_path = external_dir / "external_key_table_manifest.csv"
external_manifest.to_csv(external_manifest_path, index=False)

print(key_raw_dir)
print(external_manifest_path)

display(external_manifest)

/Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/external_sources/key_raw_tables
/Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/external_sources/external_key_table_manifest.csv


,table_name,path,rows,cols
0,atl_overall,/Users/Sumaitat/Documents/Coding/ML_LaborTight...,351,17
1,atl_overall_12ma,/Users/Sumaitat/Documents/Coding/ML_LaborTight...,352,4
2,phl_ruc,/Users/Sumaitat/Documents/Coding/ML_LaborTight...,949,243
3,phl_cpi,/Users/Sumaitat/Documents/Coding/ML_LaborTight...,949,243
4,spf_inflation,/Users/Sumaitat/Documents/Coding/ML_LaborTight...,225,5
